# 📘 Semaine 12 — TP noté n°1 (GPIO / EXTI / TIMER / PWM)

**Cours :** Microcontrôleurs STM32F103C6T6  
**Durée :** 4h30 (1h30 rappels + 1h30 TP noté + 1h30 auto-correction)  
**Enseignant :** ____________________  
**Étudiant :** ____________________  
**Binôme :** ____________________  
**Date :** ____________________

---

## 🎯 Objectifs de la semaine

1. **Réviser** les périphériques GPIO, EXTI, TIMER et PWM (S3 à S6).
2. **Démontrer** en conditions d'examen la maîtrise de CubeMX + HAL.
3. **Réaliser** un système complet en 1h30 sur carte STM32F103C6T6.
4. **Documenter** et **justifier** les choix techniques dans un compte-rendu.

---

## 🗺️ Plan de la semaine

| Séquence | Durée | Contenu |
|---|---|---|
| **A — Rappels** | 1h30 | GPIO, EXTI, TIMER, PWM (fiches + mini-exercices) |
| **B — TP noté n°1** | 1h30 | Évaluation pratique en binôme |
| **C — Auto-correction** | 1h30 | Feedback + préparation S13 |

---

## 📊 Barème du TP noté (sur 20)

| Critère | Points |
|---|---|
| Configuration CubeMX correcte | 4 |
| Code fonctionnel (compilation + exécution) | 8 |
| Respect des contraintes (fréquences, broches) | 4 |
| Propreté du code (noms, commentaires, structure) | 2 |
| **Bonus** : affichage UART | 2 |
| **Total** | **/20** |

---

# 🎓 PARTIE A — RAPPELS (1h30)

## 🔹 Fiche 1 — GPIO (10 min)

### 📖 Registres clés

| Registre | Rôle |
|---|---|
| `CRL` / `CRH` | Configuration (mode + CNF) |
| `IDR` | Lecture des entrées |
| `ODR` | Écriture des sorties |
| `BSRR` | Set/Reset atomique |

### 📖 HAL / LL / Registres

```c
// HAL
HAL_GPIO_WritePin(GPIOC, GPIO_PIN_13, GPIO_PIN_SET);
HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
GPIO_PinState s = HAL_GPIO_ReadPin(GPIOA, GPIO_PIN_0);

// LL
LL_GPIO_SetOutputPin(GPIOC, LL_GPIO_PIN_13);

// Registres
GPIOC->BSRR = GPIO_BSRR_BS13;   // Set
GPIOC->BSRR = GPIO_BSRR_BR13;   // Reset
GPIOC->ODR ^= (1 << 13);        // Toggle
```

### 📖 Pull-up vs pull-down

| Configuration | Cas d'usage |
|---|---|
| **Pull-up** | Bouton actif bas (relié au GND) |
| **Pull-down** | Bouton actif haut (relié au VCC) |
| **Flottante** | Signal piloté par un driver externe |

---

## 🔹 Fiche 2 — EXTI + NVIC (10 min)

### 📖 Chaîne EXTI

```
PA0 ──▶ AFIO_EXTICR1 ──▶ EXTI0 ──▶ Edge Detect ──▶ IMR ──▶ PR ──▶ NVIC ──▶ CPU
```

### 📖 Registres clés

| Registre | Rôle |
|---|---|
| `AFIO->EXTICR[]` | Mapping broche ↔ ligne EXTI |
| `EXTI->IMR` | Masque (1 = activée) |
| `EXTI->RTSR` | Front montant |
| `EXTI->FTSR` | Front descendant |
| `EXTI->PR` | Pending (à acquitter) |
| `NVIC->ISER[]` | Enable IRQ |
| `NVIC->IPR[]` | Priorité |

### 📖 Callback HAL

```c
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)
    {
        // Traitement EXTI0
    }
}
```

---

## 🔹 Fiche 3 — TIMER (10 min)

### 📖 Formule fondamentale

```
                  F_clk
F_timer  =  ──────────────────────
             (PSC + 1) × (ARR + 1)
```

### 📖 Timers du STM32F103C6T6

| Timer | Bus | Bits | Canaux |
|---|---|---|---|
| TIM1 | APB2 (72 MHz) | 16 | 4 + complémentaires |
| TIM2 | APB1 (72 MHz ×2) | **32** | 4 |
| TIM3 | APB1 (72 MHz ×2) | 16 | 4 |

### 📖 HAL

```c
HAL_TIM_Base_Start_IT(&htim2);

void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        // Tick périodique
    }
}
```

---

## 🔹 Fiche 4 — PWM (10 min)

### 📖 Formules

```
F_pwm  = F_clk / ((PSC + 1) × (ARR + 1))
Duty   = CCR / (ARR + 1)
CCR    = Duty × (ARR + 1)
```

### 📖 Broches PWM du STM32F103C6T6

| Timer | CH1 | CH2 | CH3 | CH4 |
|---|---|---|---|---|
| TIM1 | PA8 | PA9 | PA10 | PA11 |
| TIM2 | PA0 | PA1 | PA2 | PA3 |
| TIM3 | PA6 | PA7 | PB0 | PB1 |

### 📖 HAL

```c
HAL_TIM_PWM_Start(&htim3, TIM_CHANNEL_1);
__HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr);
```

---

## 🔹 Mini-exercices de révision (30 min)

### ✍️ Exercice A — Calculs timer/PWM

**1.** TIM2, F_clk = 72 MHz, PSC = 7199, ARR = 9999. Fréquence ? ...

**2.** TIM3, F_clk = 72 MHz, PSC = 71, ARR = 999. Fréquence ? ...

**3.** PWM 1 kHz, ARR = 999. CCR pour duty 50 % ? ...

**4.** PWM 50 Hz, ARR = 19999. CCR pour un servo à 0° (T_on = 1 ms) ? ...

**5.** PWM 50 Hz, ARR = 19999. CCR pour un servo à 180° (T_on = 2 ms) ? ...

In [ ]:
# Corrigé Exercice A
F_CLK = 72_000_000

def f_timer(psc, arr):
    return F_CLK / ((psc + 1) * (arr + 1))

print("1.", f"TIM2 PSC=7199 ARR=9999 → {f_timer(7199, 9999):.4f} Hz (≈ 1 Hz)")
print("2.", f"TIM3 PSC=71 ARR=999 → {f_timer(71, 999):.0f} Hz (1 kHz)")
print("3.", f"PWM 1 kHz ARR=999, duty=50% → CCR = {int(0.5 * 1000)}")
print("4.", f"Servo 0°: CCR = {int(1.0/20 * 20000)}  (T_on = 1 ms)")
print("5.", f"Servo 180°: CCR = {int(2.0/20 * 20000)}  (T_on = 2 ms)")

### ✍️ Exercice B — Configuration rapide

**Pour chaque ligne, indiquer la configuration CubeMX nécessaire :**

| # | Besoin | Périphérique | Broche | Paramètres |
|---|---|---|---|---|
| 1 | Bouton avec interruption | ? | ? | ? |
| 2 | LED clignote à 2 Hz | ? | ? | ? |
| 3 | PWM LED à 1 kHz | ? | ? | ? |
| 4 | Servo à 50 Hz | ? | ? | ? |
| 5 | Chronomètre à 10 ms | ? | ? | ? |

In [ ]:
# Corrigé Exercice B
configs = [
    (1, "Bouton EXTI", "EXTI0", "PA0", "Mode=IT_RISING, Pull=PULLDOWN, NVIC activé"),
    (2, "LED + TIM2 IT", "TIM2", "PC13", "PSC=7199, ARR=9999 (1 Hz à 72 MHz)"),
    (3, "PWM LED", "TIM3_CH1", "PA6", "PSC=71, ARR=999, F=1 kHz, CCR variable"),
    (4, "Servo", "TIM2_CH2", "PA1", "PSC=71, ARR=19999, F=50 Hz, CCR=1000..2000"),
    (5, "Chrono 10 ms", "TIM3 IT", "—", "PSC=719, ARR=999 (100 Hz à 72 MHz)"),
]

for c in configs:
    print(f"{c[0]}. {c[1]:<18} {c[2]:<14} {c[3]:<6} {c[4]}")

### ✍️ Exercice C — Diagnostic

**Un étudiant a configuré TIM2 (PSC=0, ARR=0) et observe un signal très rapide. Pourquoi ?**

**Réponse attendue :** ...

**Un code `HAL_GPIO_TogglePin()` dans une ISR EXTI ne semble pas fonctionner. Causes possibles ?**

**Réponse attendue :** ...

**Un chenillard basé sur `HAL_Delay(500)` est très imprécis. Pourquoi ?**

**Réponse attendue :** ...

### ✅ Corrigé Exercice C

**1. TIM2 (PSC=0, ARR=0) → très rapide**
- F = 72 MHz / (1 × 1) = **72 MHz**
- La formule nécessite toujours `(PSC+1) × (ARR+1)`. Si PSC=0 et ARR=0, le timer déborde à chaque cycle.
- Solution : utiliser PSC=7199, ARR=9999 pour 1 Hz.

**2. Toggle dans ISR qui ne marche pas**
- Pending non acquitté (si accès registres directs)
- Variable modifiée non déclarée `volatile`
- NVIC non activé
- Priorité bloquée par une autre ISR plus prioritaire
- Rebonds non filtrés (comptage multiple)

**3. HAL_Delay imprécis**
- Basé sur SysTick (interruption toutes les 1 ms)
- Erreur possible si SysTick est désactivé ou si le CPU est surchargé
- `HAL_Delay` bloque le CPU
- Pour une cadence précise : **utiliser un TIMER matériel**.

---

## 🔹 QCM de révision (10 min)

**1. Combien de broches par port GPIO ?**  
A. 8  B. 16  C. 32  D. 64

**2. Le registre qui active une ligne EXTI est :**  
A. IMR  B. RTSR  C. PR  D. SWIER

**3. Pour TIM2, PSC = 71, ARR = 999 (72 MHz), la fréquence est :**  
A. 100 Hz  B. 1 kHz  C. 10 kHz  D. 100 kHz

**4. En PWM mode 1, la sortie est active tant que :**  
A. CNT < CCR  B. CNT ≥ CCR  C. CNT = 0  D. CNT = ARR

**5. Combien de niveaux de priorité NVIC ?**  
A. 4  B. 8  C. 16  D. 256

**6. La fonction HAL pour démarrer un canal PWM est :**  
A. `HAL_TIM_Base_Start()`  B. `HAL_TIM_PWM_Start()`  C. `HAL_GPIO_WritePin()`  D. `HAL_ADC_Start()`

### ✅ Corrigé QCM

| Q | Rép. |
|---|---|
| 1 | **B — 16** |
| 2 | **A — IMR** |
| 3 | **B — 1 kHz** |
| 4 | **A — CNT < CCR** |
| 5 | **C — 16** |
| 6 | **B — `HAL_TIM_PWM_Start()`** |

---

# 🛠️ PARTIE B — TP NOTÉ N°1 (1h30)

## 📋 Consignes générales

- **Durée : 1h30** (strict).
- **En binôme**. Un seul rendu par binôme.
- **Documents autorisés** : notes personnelles, datasheet, RM0008 (format papier).
- **Sans échange entre binômes.**
- **Aucun accès internet.**
- **Livrables** : projet CubeIDE + capture écran + compte-rendu.

---

## 📄 Sujet du TP noté n°1

### 🎯 Objectif
Réaliser un système complet de contrôle de LEDs avec boutons et PWM sur carte **STM32F103C6T6**.

### 📖 Cahier des charges

**1. LED principale (PC13)**
- Clignote à **2 Hz** (période 500 ms) via **TIM2** en mode interruption.
- Fréquence modifiable par bouton (voir §3).

**2. Boutons**
- **PA0 (EXTI0, front montant, pull-down)** : change la fréquence du clignotement.
  - Cycle entre 1 Hz → 2 Hz → 4 Hz → 1 Hz …
- **PA1 (EXTI1, front montant, pull-down)** : reset.
  - Remet la fréquence à 2 Hz et le PWM à 50 %.

**3. PWM (PA6, TIM3_CH1)**
- Fréquence PWM : **1 kHz** (PSC=71, ARR=999).
- Le duty cycle varie proportionnellement au nombre d'appuis sur PA0 :
  - 0 appui : 0 %
  - 1 appui : 33 %
  - 2 appuis : 66 %
  - 3 appuis : 100 %
  - 4 appuis : retour à 0 %

**4. Affichage UART2 (bonus)**
- Envoyer sur **USART2 (115200 bauds)** à chaque changement :
  - La fréquence de clignotement actuelle
  - La valeur de duty PWM

### 📊 Barème détaillé

| Critère | Points | Détail |
|---|---|---|
| **Configuration CubeMX** | **4** | TIM2 IT, EXTI0, EXTI1, PWM TIM3_CH1, USART2, GPIO |
| **Code fonctionnel** | **8** | LED clignote, boutons réactifs, PWM varie |
| **Respect contraintes** | **4** | Fréquences exactes, broches respectées |
| **Propreté du code** | **2** | Nommage, commentaires, structure |
| **Bonus UART** | **+2** | Format clair et lisible |
| **Total** | **/20** | |

---

## 📖 Aide-mémoire autorisé

### Formules

```
TIM2 (1 Hz)  : PSC = 7199, ARR = 9999
TIM2 (2 Hz)  : PSC = 7199, ARR = 4999
TIM2 (4 Hz)  : PSC = 7199, ARR = 2499

TIM3 PWM 1 kHz : PSC = 71, ARR = 999
  CCR = 0    → duty   0 %
  CCR = 333  → duty  33 %
  CCR = 666  → duty  66 %
  CCR = 999  → duty 100 %
```

### Squelette de code

```c
/* --- Variables globales --- */
volatile uint8_t  etape_freq = 0;   // 0=1Hz, 1=2Hz, 2=4Hz
volatile uint8_t  etape_pwm  = 0;   // 0..3
volatile uint8_t  appuis_pa0 = 0;
volatile uint8_t  tick_led   = 0;
volatile uint8_t  diviseur   = 0;

/* --- Table des fréquences --- */
static const uint16_t arr_freq[3] = { 9999, 4999, 2499 };

/* --- Callback TIM2 (base de temps 1 Hz) --- */
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        uint8_t cible = (etape_freq == 0) ? 1 : (etape_freq == 1 ? 2 : 4);
        diviseur = (diviseur + 1) % cible;
        if (diviseur == 0)
            HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
    }
}

/* --- Callback EXTI --- */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)   // PA0
    {
        // 1) Fréquence
        etape_freq = (etape_freq + 1) % 3;
        __HAL_TIM_SET_AUTORELOAD(&htim2, arr_freq[etape_freq]);

        // 2) PWM
        etape_pwm = (etape_pwm + 1) % 4;
        uint16_t ccr = (uint16_t)(etape_pwm * 999 / 3);
        __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr);
    }
    else if (GPIO_Pin == GPIO_PIN_1)  // PA1
    {
        etape_freq = 1;
        etape_pwm  = 1;
        __HAL_TIM_SET_AUTORELOAD(&htim2, arr_freq[1]);
        __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, 333);
    }
}
```

### Configuration CubeMX

| Périphérique | Config |
|---|---|
| PC13 | GPIO_Output |
| PA6 | TIM3_CH1 (PWM Generation CH1) |
| PA0 | GPIO_EXTI0 (Rising edge, Pull-down) |
| PA1 | GPIO_EXTI1 (Rising edge, Pull-down) |
| TIM2 | PSC=7199, ARR=9999, NVIC activé |
| TIM3 | PSC=71, ARR=999, PWM Mode 1 |
| USART2 | 115200, 8N1, PA2/PA3 |

---

## 📝 Feuille de route (par binôme)

| Étape | Durée cible | Cocher |
|---|---|---|
| Créer le projet CubeIDE `TP_NOT1_NOM` | 5 min | ☐ |
| Configurer les broches et périphériques | 20 min | ☐ |
| Générer le code et écrire les callbacks | 30 min | ☐ |
| Compiler et flasher | 5 min | ☐ |
| Tester et déboguer | 15 min | ☐ |
| Ajouter UART (bonus) | 10 min | ☐ |
| Remplir le compte-rendu | 5 min | ☐ |
| **Total** | **1h30** | |

---

## 📋 Compte-rendu de TP noté n°1

**Nom 1 :** __________________  **Nom 2 :** __________________  
**Date :** __________________  **Groupe :** __________________

### 1. Configuration CubeMX (4 pts)

| Périphérique | Broche | Configuration |
|---|---|---|
| LED | ... | ... |
| Bouton fréquence | ... | ... |
| Bouton reset | ... | ... |
| PWM | ... | ... |
| UART (bonus) | ... | ... |

**Captures d'écran :** ☐ Pinout  ☐ Clock Tree  ☐ NVIC  ☐ TIM2  ☐ TIM3

### 2. Code fonctionnel (8 pts)

**Fonctionnalités implémentées :**
- ☐ LED clignote sur PC13
- ☐ Fréquence change avec PA0
- ☐ Reset avec PA1
- ☐ PWM varie sur PA6

**Extrait du code ajouté :**
```c
// Coller ici le code
```

### 3. Respect des contraintes (4 pts)

| Contrainte | Valeur attendue | Valeur mesurée | Conforme |
|---|---|---|---|
| Fréquence LED initiale | 2 Hz | ... | ☐ |
| Fréquence après PA0 | 4 Hz | ... | ☐ |
| Fréquence PWM | 1 kHz | ... | ☐ |
| Duty 33 % | CCR = 333 | ... | ☐ |
| Duty 100 % | CCR = 999 | ... | ☐ |

### 4. Propreté du code (2 pts)

- ☐ Noms de variables clairs
- ☐ Commentaires présents
- ☐ Structure logique
- ☐ Variables partagées en `volatile`

### 5. Bonus UART (2 pts)

**Format envoyé :**
```
Freq = 2 Hz, Duty = 33 %
```

☐ Format clair  ☐ Envoyé à chaque changement

### 6. Difficultés rencontrées

- ...

### 7. Solutions apportées

- ...

### 8. Auto-évaluation

| Critère | Note estimée |
|---|---|
| Configuration CubeMX | /4 |
| Code fonctionnel | /8 |
| Respect contraintes | /4 |
| Propreté | /2 |
| Bonus UART | /2 |
| **Total** | **/20** |

---

# 🎓 PARTIE C — AUTO-CORRECTION & PRÉPARATION S13 (1h30)

## 📖 Corrigé type — Solution complète

In [ ]:
/* ============================================================
   CORRIGÉ TP NOTÉ N°1 — Code complet
   ============================================================ */

#include "main.h"
#include <stdio.h>

TIM_HandleTypeDef htim2;
TIM_HandleTypeDef htim3;
UART_HandleTypeDef huart2;

/* --- Tables --- */
static const uint16_t arr_freq[3] = { 9999, 4999, 2499 };   // 1 Hz, 2 Hz, 4 Hz
static const uint8_t  freq_hz[3]  = { 1, 2, 4 };
static const uint16_t ccr_pwm[4]  = { 0, 333, 666, 999 };
static const uint8_t  duty_pct[4] = { 0, 33, 66, 100 };

/* --- Variables globales --- */
volatile uint8_t etape_freq = 1;   // démarrer à 2 Hz
volatile uint8_t etape_pwm  = 1;   // démarrer à 33 %
volatile uint8_t diviseur   = 0;

/* --- Envoi UART du statut --- */
static void envoyer_statut(void)
{
    char buf[64];
    int n = snprintf(buf, sizeof(buf),
                     "Freq = %d Hz, Duty = %d %%\r\n",
                     freq_hz[etape_freq], duty_pct[etape_pwm]);
    HAL_UART_Transmit(&huart2, (uint8_t*)buf, n, 100);
}

/* --- Callback TIM2 --- */
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2)
    {
        uint8_t cible = freq_hz[etape_freq];
        diviseur = (diviseur + 1) % cible;
        if (diviseur == 0)
            HAL_GPIO_TogglePin(GPIOC, GPIO_PIN_13);
    }
}

/* --- Callback EXTI --- */
void HAL_GPIO_EXTI_Callback(uint16_t GPIO_Pin)
{
    if (GPIO_Pin == GPIO_PIN_0)
    {
        etape_freq = (etape_freq + 1) % 3;
        etape_pwm  = (etape_pwm + 1) % 4;
        __HAL_TIM_SET_AUTORELOAD(&htim2, arr_freq[etape_freq]);
        __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr_pwm[etape_pwm]);
        envoyer_statut();
    }
    else if (GPIO_Pin == GPIO_PIN_1)
    {
        etape_freq = 1;
        etape_pwm  = 1;
        __HAL_TIM_SET_AUTORELOAD(&htim2, arr_freq[etape_freq]);
        __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr_pwm[etape_pwm]);
        envoyer_statut();
    }
}

int main(void)
{
    HAL_Init();
    SystemClock_Config();
    MX_GPIO_Init();
    MX_TIM2_Init();
    MX_TIM3_Init();
    MX_USART2_UART_Init();

    HAL_TIM_Base_Start_IT(&htim2);
    HAL_TIM_PWM_Start(&htim3, TIM_CHANNEL_1);

    // Initialiser PWM à 33 %
    __HAL_TIM_SET_COMPARE(&htim3, TIM_CHANNEL_1, ccr_pwm[etape_pwm]);

    while (1)
    {
        // Rien : tout est géré par les interruptions
    }
}

### 📊 Grille de correction officielle (enseignant)

| Critère | 0 pt | 2 pts | 4 pts |
|---|---|---|---|
| **Configuration CubeMX** | Rien | Partielle | Complète et correcte |
| **Code fonctionnel** | Ne compile pas | Fonctionne partiellement | Fonctionne entièrement |
| **Respect contraintes** | Fréquences fausses | 1-2 erreurs | Toutes correctes |
| **Propreté** | Code illisible | Moyen | Clair et commenté |

**Code fonctionnel (8 pts) — Détail :**
- LED clignote via TIM2 : 2 pts
- PA0 change la fréquence : 2 pts
- PA1 reset : 1 pt
- PWM varie sur PA6 : 2 pts
- Anti-rebond PA0/PA1 : 1 pt

---

## 🐍 Simulation Python — Correction automatique

Utilise cette cellule pour simuler le comportement attendu et vérifier une implémentation.

In [ ]:
# ============================================================
# Simulation du comportement attendu — TP noté n°1
# ============================================================

class SystemeTP:
    def __init__(self):
        self.etape_freq = 1        # 0=1Hz, 1=2Hz, 2=4Hz
        self.etape_pwm  = 1        # 0..3
        self.freq_hz    = [1, 2, 4]
        self.arr_freq   = [9999, 4999, 2499]
        self.ccr_pwm    = [0, 333, 666, 999]
        self.duty_pct   = [0, 33, 66, 100]
        self.journal    = []

    def appui_pa0(self):
        self.etape_freq = (self.etape_freq + 1) % 3
        self.etape_pwm  = (self.etape_pwm + 1) % 4
        self._log("PA0")

    def appui_pa1(self):
        self.etape_freq = 1
        self.etape_pwm  = 1
        self._log("PA1 (reset)")

    def _log(self, source):
        self.journal.append(
            f"{source:<15} Freq={self.freq_hz[self.etape_freq]} Hz, "
            f"Duty={self.duty_pct[self.etape_pwm]} % (CCR={self.ccr_pwm[self.etape_pwm]})"
        )

    def afficher(self):
        print("État actuel :")
        print(f"  Fréquence LED : {self.freq_hz[self.etape_freq]} Hz")
        print(f"  ARR TIM2      : {self.arr_freq[self.etape_freq]}")
        print(f"  Duty PWM      : {self.duty_pct[self.etape_pwm]} %")
        print(f"  CCR TIM3_CH1  : {self.ccr_pwm[self.etape_pwm]}")

# Scénario de test
sys = SystemeTP()
sys.afficher()
print()

sys.appui_pa0(); sys.appui_pa0(); sys.appui_pa0()
sys.appui_pa1()
sys.appui_pa0()

print("📋 Journal des événements :")
for ligne in sys.journal:
    print(f"  {ligne}")

print()
sys.afficher()

---

## 📊 Grille d'auto-évaluation

Coche chaque point validé par ton binôme.

### Configuration CubeMX (4 pts)

- [ ] PC13 configuré en GPIO_Output
- [ ] PA0 configuré en GPIO_EXTI0 (front montant, pull-down)
- [ ] PA1 configuré en GPIO_EXTI1 (front montant, pull-down)
- [ ] PA6 configuré en TIM3_CH1 (PWM Generation)
- [ ] TIM2 avec PSC=7199 et ARR initial = 4999 (2 Hz)
- [ ] NVIC : TIM2 global interrupt activé
- [ ] NVIC : EXTI0 et EXTI1 activés
- [ ] USART2 configuré (bonus)

### Code fonctionnel (8 pts)

- [ ] `HAL_TIM_Base_Start_IT(&htim2)` appelé
- [ ] `HAL_TIM_PWM_Start(&htim3, TIM_CHANNEL_1)` appelé
- [ ] Callback `HAL_TIM_PeriodElapsedCallback` implémenté
- [ ] Callback `HAL_GPIO_EXTI_Callback` implémenté
- [ ] LED toggle sur PC13 dans le callback timer
- [ ] Changement ARR sur appui PA0
- [ ] Changement CCR sur appui PA0
- [ ] Reset sur PA1

### Respect contraintes (4 pts)

- [ ] Fréquence initiale = 2 Hz
- [ ] Fréquence suit 1→2→4 Hz
- [ ] PWM à 1 kHz (PSC=71, ARR=999)
- [ ] Duty varie 0→33→66→100 %

### Propreté (2 pts)

- [ ] Variables `volatile` pour les partagées
- [ ] Code commenté et structuré

### Bonus UART (2 pts)

- [ ] Message envoyé à chaque changement
- [ ] Format clair et lisible

---

## 🔍 Erreurs fréquentes et solutions

| Erreur | Cause probable | Solution |
|---|---|---|
| LED ne clignote pas | `HAL_TIM_Base_Start_IT()` oublié | Ajouter l'appel dans `main()` |
| Fréquence ne change pas | ARR modifié mais pas de `UG` event | Utiliser `__HAL_TIM_SET_AUTORELOAD` + `TIM_EGR_UG` |
| PWM bloqué | `HAL_TIM_PWM_Start()` oublié | Ajouter l'appel |
| Rebonds boutons | Absence d'anti-rebond | Ajouter `HAL_Delay(20)` + relâchement |
| ISR non déclenchée | NVIC non activé dans CubeMX | Cocher la case dans NVIC Settings |
| Variable non mise à jour | Manque de `volatile` | Ajouter `volatile` à la déclaration |
| UART ne transmet rien | Baud rate incorrect | Vérifier BRR et câblage TX/RX |

---

## 🎯 Objectifs de la S13

### 📋 Programme du TP noté n°2

- **Acquisition ADC** sur PA0 via DMA
- **Déclenchement par TIM2** à 1 kHz
- **Moyenne** sur 100 échantillons
- **Détection de seuil** (2 V) avec hystérésis
- **Envoi UART** des données à 10 Hz
- **Bonus** : lecture d'un capteur I2C (température)

### 📖 Préparation

- Revoir les notebooks **S8 (ADC)** et **S9 (ADC + DMA)**.
- S'entraîner à configurer ADC + DMA en CubeMX.
- Réviser le filtrage numérique (moyenne, médiane, IIR).
- Préparer les questions sur les callbacks DMA.

---

## 📊 Bilan de la semaine 12

| Item | Statut |
|---|---|
| Révisions A (fiches + exercices) | ☐ |
| TP noté n°1 (1h30) | ☐ |
| Compte-rendu rendu | ☐ |
| Auto-évaluation remplie | ☐ |
| Préparation S13 | ☐ |

### 📈 Note estimée du TP noté : ___ / 20

| Score | Interprétation |
|---|---|
| 16–20 | ✅ Excellent |
| 12–15 | ✅ Bien |
| 8–11 | ⚠️ Passable — revoir les bases |
| < 8 | ❌ Insuffisant — rattrapage S13 |

---
# 📚 RESSOURCES Semaine 12

### Documents officiels
- 📄 **RM0008** — chapitres 9 (GPIO), 10 (EXTI), 15 (TIM), 16 (TIM généraux)
- 📄 **Datasheet STM32F103x6**
- 📄 **UM1850** — HAL documentation

### Notebooks de référence
- 📓 S3 — GPIO (HAL/LL/Registres)
- 📓 S4 — EXTI + NVIC
- 📓 S5 — TIMER (base de temps)
- 📓 S6 — PWM

### Outils
- **STM32CubeIDE**
- **STM32CubeMX**
- **ST-Link V2**
- **Oscilloscope** (pour vérifier les fréquences)
- **Terminal série** (pour le bonus UART)

---

**Fin du notebook — Semaine 12** ✨